In [1]:
import pandas as pd
# 데이터 로드
df_bus = pd.read_csv('단위면적당 버스정류장 수.csv')
df_con = pd.read_csv('소비데이터.csv')
df_safe = pd.read_csv('시군구_안전지표.csv')
df_eat = pd.read_csv('식당 평점 및 리뷰 수.csv')

In [2]:
# 제주도 데이터 추가
df_safe.loc[226] = ['제주도','제주시', 2, 2, 5, 5, 3, 2]
df_safe.loc[227] = ['제주도','서귀포시', 2, 2, 5, 5, 3, 2]
df_safe

,시도,시군구,교통사고,화재,범죄,생활안전,자살,감염병
0,경기,수원시,1,1.0,4,2,2,2
1,경기,성남시,2,1.0,4,1,2,3
2,경기,의정부시,1,2.0,4,2,3,4
3,경기,안양시,2,1.0,3,1,2,3
4,경기,부천시,2,2.0,5,2,2,3
...,...,...,...,...,...,...,...,...
223,울산,남구,2,1.0,4,5,4,2
224,울산,동구,1,2.0,1,4,3,1
225,울산,북구,2,1.0,1,2,2,2
226,제주도,제주시,2,2.0,5,5,3,2


In [3]:
import numpy as np
import pandas as pd

# 식당 평가 지표 (평점, 방문자 리뷰 수, 블로그 리뷰 수)
restaurant_criteria = np.array([4.08, 3.83, 2.41])

# 쌍대 비교 행렬 생성
restaurant_matrix = np.array([
    [restaurant_criteria[0] / restaurant_criteria[0], restaurant_criteria[0] / restaurant_criteria[1], restaurant_criteria[0] / restaurant_criteria[2]],
    [restaurant_criteria[1] / restaurant_criteria[0], restaurant_criteria[1] / restaurant_criteria[1], restaurant_criteria[1] / restaurant_criteria[2]],
    [restaurant_criteria[2] / restaurant_criteria[0], restaurant_criteria[2] / restaurant_criteria[1], restaurant_criteria[2] / restaurant_criteria[2]]
])

# 각 행의 합 계산
restaurant_row_sums = restaurant_matrix.sum(axis=1)

# 전체 합 계산
total_sum = restaurant_row_sums.sum()

# 가중치 계산 (각 행의 합을 전체 합으로 나누기)
restaurant_weights = restaurant_row_sums / total_sum

# 결과 출력
print("식당 평가 가중치:", restaurant_weights)


식당 평가 가중치: [0.39534884 0.37112403 0.23352713]


In [4]:
# 식당 평가지표 생성
# 매핑 딕셔너리 생성
mapping = {
    '강원특별자치도': '강원',
    '경기도': '경기',
    '경상남도': '경남',
    '경상북도': '경북',
    '광주광역시': '광주',
    '대구광역시': '대구',
    '대전광역시': '대전',
    '부산광역시': '부산',
    '서울특별시': '서울',
    '울산광역시': '울산',
    '인천광역시': '인천',
    '전라남도': '전남',
    '전북특별자치도': '전북',
    '제주특별자치도': '제주도',
    '충청남도': '충남',
    '충청북도': '충북'
}

# 첫 번째 데이터프레임의 '시도명' 컬럼 값 치환
df_eat['시도명'] = df_eat['시도명'].replace(mapping)
df_eat['지역명'] = df_eat.loc[:,'시도명'] + ' '+ df_eat.loc[:,'시군구']
df_eat

# 가중치 값 정의
weight_rating = 0.3954
weight_visitor_reviews = 0.3711
weight_blog_reviews = 0.2335

# 각 가중치를 곱한 값을 합산하여 새로운 열 추가
df_eat['식당 평가 지표'] = (df_eat['평점'] * weight_rating) + (df_eat['방문자 리뷰 수'] * weight_visitor_reviews) + (df_eat['블로그 리뷰 수'] * weight_blog_reviews)
df_eat['지역명'] = df_eat.loc[:,'시도명'] + ' '+ df_eat.loc[:,'시군구']
df_eat = df_eat.drop(index=227)
df_eat

,시도명,시군구,평점,방문자 리뷰 수,블로그 리뷰 수,지역명,식당 평가 지표
0,강원,강릉시,4.385164,1017.639344,262.049180,강원 강릉시,440.568338
1,강원,고성군,4.394643,559.571429,334.178571,강원 고성군,287.425295
2,강원,동해시,4.363016,754.555556,182.142857,강원 동해시,324.271060
3,강원,삼척시,4.280000,560.245614,128.736842,강원 삼척시,239.659512
4,강원,속초시,4.348367,1626.469388,790.836735,강원 속초시,789.962512
...,...,...,...,...,...,...,...
223,충북,진천군,4.343409,532.522727,39.613636,충북 진천군,208.586352
224,충북,청주시,4.417287,891.251012,72.643725,충북 청주시,349.452156
225,충북,충주시,4.367285,468.152318,54.337748,충북 충주시,188.146014
226,서울,도봉구,4.368846,1479.384615,52.384615,서울 도봉구,562.958880


In [5]:
df = df_eat[['지역명','식당 평가 지표']]
df

,지역명,식당 평가 지표
0,강원 강릉시,440.568338
1,강원 고성군,287.425295
2,강원 동해시,324.271060
3,강원 삼척시,239.659512
4,강원 속초시,789.962512
...,...,...
223,충북 진천군,208.586352
224,충북 청주시,349.452156
225,충북 충주시,188.146014
226,서울 도봉구,562.958880


In [6]:
# 지역별 교통접근성 평가지표 생성
df_bus = df_bus.rename(columns={'단위면적당 버스정류장 수':'교통접근성 평가 지표'})

# 첫 번째 데이터프레임의 '시도명' 컬럼 값 치환
df_bus['시도명'] = df_bus['시도명'].replace(mapping)
df_bus['지역명'] = df_bus.loc[:,'시도명'] + ' '+ df_bus.loc[:,'시군구']
df_bus

,시도명,시군구,버스정류장 수,면적(km^2),교통접근성 평가 지표,지역명
0,강원,강릉시,38,1040,0.036538,강원 강릉시
1,강원,고성군,11,660,0.016667,강원 고성군
2,강원,동해시,393,180,2.183333,강원 동해시
3,강원,봉화군,6,1202,0.004992,강원 봉화군
4,강원,삼척시,141,1187,0.118787,강원 삼척시
...,...,...,...,...,...,...
235,충북,제천시,1483,882,1.681406,충북 제천시
236,충북,증평군,68,81,0.839506,충북 증평군
237,충북,진천군,751,407,1.845209,충북 진천군
238,충북,청주시,3465,941,3.682253,충북 청주시


In [7]:
# '시군구'가 중복되는 경우 확인 (중복된 '시군구'를 기준으로 처리)  => 이거 내일 확인하고 지워야하는거 인덱스 적어놓기: 193, 194,77, 3, 8, 231, 83, 9,222, 223, 64, 208
duplicate_rows = df_bus[df_bus.duplicated(subset=['시군구'],keep=False)].sort_values(by='시군구')
duplicate_rows

,시도명,시군구,버스정류장 수,면적(km^2),교통접근성 평가 지표,지역명
133,서울,강서구,257,41,6.268293,서울 강서구
114,부산,강서구,1543,41,37.634146,부산 강서구
1,강원,고성군,11,660,0.016667,강원 고성군
53,경남,고성군,893,660,1.353030,경남 고성군
193,전북,곡성군,2,547,0.003656,전북 곡성군
172,전남,곡성군,536,547,0.979890,전남 곡성군
228,충북,괴산군,948,842,1.125891,충북 괴산군
73,경북,괴산군,24,842,0.028504,경북 괴산군
194,전북,구례군,1,442,0.002262,전북 구례군
174,전남,구례군,419,442,0.947964,전남 구례군


In [8]:
# 지우고 싶은 행의 인덱스 리스트
indices_to_drop = [193, 194, 77, 3, 8, 231, 83, 9, 222, 223, 64, 208]

# 해당 인덱스를 드랍
df_bus = df_bus.drop(index=indices_to_drop)
df_bus

,시도명,시군구,버스정류장 수,면적(km^2),교통접근성 평가 지표,지역명
0,강원,강릉시,38,1040,0.036538,강원 강릉시
1,강원,고성군,11,660,0.016667,강원 고성군
2,강원,동해시,393,180,2.183333,강원 동해시
4,강원,삼척시,141,1187,0.118787,강원 삼척시
5,강원,속초시,204,105,1.942857,강원 속초시
...,...,...,...,...,...,...
235,충북,제천시,1483,882,1.681406,충북 제천시
236,충북,증평군,68,81,0.839506,충북 증평군
237,충북,진천군,751,407,1.845209,충북 진천군
238,충북,청주시,3465,941,3.682253,충북 청주시


In [9]:
# 치안 평가 지표 (교통사고, 화재, 범죄, 생활안전, 자살, 감염병)
safety_criteria = np.array([2.34, 1.98, 4.48, 3.3, 1.68, 3.31])

# 쌍대 비교 행렬 생성
safety_matrix = np.array([
    [safety_criteria[0] / safety_criteria[0], safety_criteria[0] / safety_criteria[1], safety_criteria[0] / safety_criteria[2], safety_criteria[0] / safety_criteria[3], safety_criteria[0] / safety_criteria[4], safety_criteria[0] / safety_criteria[5]],
    [safety_criteria[1] / safety_criteria[0], safety_criteria[1] / safety_criteria[1], safety_criteria[1] / safety_criteria[2], safety_criteria[1] / safety_criteria[3], safety_criteria[1] / safety_criteria[4], safety_criteria[1] / safety_criteria[5]],
    [safety_criteria[2] / safety_criteria[0], safety_criteria[2] / safety_criteria[1], safety_criteria[2] / safety_criteria[2], safety_criteria[2] / safety_criteria[3], safety_criteria[2] / safety_criteria[4], safety_criteria[2] / safety_criteria[5]],
    [safety_criteria[3] / safety_criteria[0], safety_criteria[3] / safety_criteria[1], safety_criteria[3] / safety_criteria[2], safety_criteria[3] / safety_criteria[3], safety_criteria[3] / safety_criteria[4], safety_criteria[3] / safety_criteria[5]],
    [safety_criteria[4] / safety_criteria[0], safety_criteria[4] / safety_criteria[1], safety_criteria[4] / safety_criteria[2], safety_criteria[4] / safety_criteria[3], safety_criteria[4] / safety_criteria[4], safety_criteria[4] / safety_criteria[5]],
    [safety_criteria[5] / safety_criteria[0], safety_criteria[5] / safety_criteria[1], safety_criteria[5] / safety_criteria[2], safety_criteria[5] / safety_criteria[3], safety_criteria[5] / safety_criteria[4], safety_criteria[5] / safety_criteria[5]]
])

# 각 행의 합 계산
safety_row_sums = safety_matrix.sum(axis=1)

# 전체 합 계산
total_sum_safety = safety_row_sums.sum()

# 가중치 계산 (각 행의 합을 전체 합으로 나누기)
safety_weights = safety_row_sums / total_sum_safety

# 결과 출력
print("치안 평가 가중치:", safety_weights)

치안 평가 가중치: [0.13692218 0.11585723 0.2621416  0.19309538 0.0983031  0.19368051]


In [10]:
weights = {
    '교통사고': 0.1368,
    '화재': 0.1158,
    '범죄': 0.2621,
    '생활안전': 0.1932,
    '자살': 0.0983,
    '감염병': 0.1938
}

# 치안 점수를 계산하여 새로운 열 추가
df_safe['치안 평가 지표'] = (df_safe['교통사고'] * weights['교통사고'] +
                df_safe['화재'] * weights['화재'] +
                df_safe['범죄'] * weights['범죄'] +
                df_safe['생활안전'] * weights['생활안전'] +
                df_safe['자살'] * weights['자살'] +
                df_safe['감염병'] * weights['감염병'])

df_safe['지역명'] = df_safe.loc[:,'시도'] + ' '+ df_safe.loc[:,'시군구']
df_safe

,시도,시군구,교통사고,화재,범죄,생활안전,자살,감염병,치안 평가 지표,지역명
0,경기,수원시,1,1.0,4,2,2,2,2.2716,경기 수원시
1,경기,성남시,2,1.0,4,1,2,3,2.4090,경기 성남시
2,경기,의정부시,1,2.0,4,2,3,4,2.8733,경기 의정부시
3,경기,안양시,2,1.0,3,1,2,3,2.1469,경기 안양시
4,경기,부천시,2,2.0,5,2,2,3,2.9801,경기 부천시
...,...,...,...,...,...,...,...,...,...,...
223,울산,남구,2,1.0,4,5,4,2,3.1846,울산 남구
224,울산,동구,1,2.0,1,4,3,1,1.8920,울산 동구
225,울산,북구,2,1.0,1,2,2,2,1.6221,울산 북구
226,제주도,제주시,2,2.0,5,5,3,2,3.4642,제주도 제주시


In [11]:
df_con = df_con.rename(columns={'관광 소비액':'관광 소비 지표'})
df_con['지역명'] = df_con.loc[:,'시도명'] + ' '+ df_con.loc[:,'시군구']
df_con = df_con.drop_duplicates(subset=['지역명'])
df_con

,시도명,시군구,관광 소비 지표,지역명
0,서울,강남구,14933.0,서울 강남구
1,서울,강동구,1950.0,서울 강동구
2,서울,강북구,1387.0,서울 강북구
3,서울,강서구,7451.0,서울 강서구
4,서울,관악구,2184.0,서울 관악구
...,...,...,...,...
224,경남,함안군,277.0,경남 함안군
225,경남,함양군,104.0,경남 함양군
226,경남,합천군,147.0,경남 합천군
227,제주도,서귀포시,3213.0,제주도 서귀포시


In [12]:
df = df.merge(df_con[['지역명','관광 소비 지표']], on='지역명',how='left')
df = df.merge(df_safe[['지역명','치안 평가 지표']], on='지역명',how='left')
df = df.merge(df_bus[['지역명','교통접근성 평가 지표']], on='지역명',how='left')
df

,지역명,식당 평가 지표,관광 소비 지표,치안 평가 지표,교통접근성 평가 지표
0,강원 강릉시,440.568338,1866.0,3.7385,0.036538
1,강원 고성군,287.425295,530.0,3.7772,0.016667
2,강원 동해시,324.271060,393.0,3.4289,2.183333
3,강원 삼척시,239.659512,507.0,3.3425,0.118787
4,강원 속초시,789.962512,1655.0,3.9321,1.942857
...,...,...,...,...,...
223,충북 진천군,208.586352,526.0,2.6908,1.845209
224,충북 청주시,349.452156,1360.5,NaN,3.682253
225,충북 충주시,188.146014,1240.0,3.4283,1.767040
226,서울 도봉구,562.958880,914.0,2.7175,4.250000


In [13]:
df.to_csv('군집분석용_데이터셋.csv',index=False)